# Notebook 11 — MNAR Extension
**Novelty:** Extending MIGA to Missing Not At Random (MNAR) mechanisms.

## Background

The base paper evaluates MIGA exclusively under **MAR** (Missing At Random),
where the probability that a value is missing is independent of the value
itself.  In many real-world settings, missingness is **not** random:

- High-income respondents may not report income (finance, surveys)
- Extreme sensor readings may be truncated or fail to record (IoT, medicine)
- Patients with severe outcomes may drop out of clinical trials

This is formally called **MNAR** (Missing Not At Random; Rubin 1976):
> $P(R=0 \mid Y) \neq P(R=0)$,  where $R$ is the response indicator.

**Key insight:** MIGA minimises the *distributional distance* between
available rows X_A and completed rows X_C — it never assumes MAR.
We test whether this distributional objective makes MIGA more robust to
MNAR than methods that explicitly assume MAR (e.g. mean imputation).

## MNAR Mechanisms Evaluated

| Mechanism | Rule | Real-world analogy |
|-----------|------|--------------------|
| MAR       | rows missing at random (baseline) | survey non-response |
| top       | top pct% of values go missing | self-censoring of high values |
| bottom    | bottom pct% of values go missing | floor effects, detection limits |
| tails     | pct/2% from each tail go missing | censoring of extreme values |

All mechanisms remove the same percentage pct% per selected feature —
only *which* rows are selected differs.

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from miga import MIGA
from miga.fitness import FitnessEvaluator
from miga.statistics import compute_stats, pooled_std, relative_cov, minkowski_distance
from miga.data_utils import apply_mar, apply_mnar, auto_generators, compute_metrics, EXCLUDE_COLS
from miga.paper_results import (
    TABLE3_PARAMS, BENCHMARK_Q,
    TABLE4_RMSE, TABLE5_MAD, TABLE6_COD,
    METHODS, PERCENTAGES,
)

RESULTS_DIR = os.path.join("..", "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Setup complete.")

## 1. Configuration

In [ ]:
from miga.data_utils import load_iris, load_glass, load_haberman

DATASET_CFG = {
    "Iris":     (load_iris,     {"pct": 30, "seed": 30, "exclude_cols": []}),
    "Glass":    (load_glass,    {"pct": 30, "seed": 30, "exclude_cols": [9]}),
    "Haberman": (load_haberman, {"pct": 30, "seed": 30, "exclude_cols": []}),
}

L, G_GENS, Q_RUNS, SEED = 200, 500, 5, 42
MECHANISMS = ["mar", "top", "bottom", "tails"]

print(f"GA: l={L}, G={G_GENS}, Q={Q_RUNS}, seed={SEED}")
print(f"Missing: 30% per selected feature")
print(f"Mechanisms: {MECHANISMS}")

## 2. MNAR Mechanism Visualisation

In [ ]:
# Show which rows each mechanism selects for a single feature of Iris
from miga.data_utils import load_iris, apply_mar, apply_mnar

X_iris, cols_iris = load_iris()
petal_len = X_iris[:, 2]  # feature 2: petal length
n = len(petal_len)
n_remove = max(1, int(round(n * 30 / 100)))

order = np.argsort(petal_len)
half  = max(1, n_remove // 2)

row_sets = {
    "MAR":    np.random.default_rng(30).choice(n, size=n_remove, replace=False),
    "top":    order[-n_remove:],
    "bottom": order[:n_remove],
    "tails":  np.concatenate([order[:half], order[-half:]]),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharey=True)
colours = {"MAR": "tab:gray", "top": "tab:red", "bottom": "tab:blue", "tails": "tab:purple"}

for ax, (mech, rows) in zip(axes.flat, row_sets.items()):
    ax.scatter(range(n), np.sort(petal_len), s=15, color="lightgray", label="observed")
    ax.scatter(rows, np.sort(petal_len)[np.argsort(np.argsort(petal_len))[rows]],
               s=25, color=colours[mech], label="missing")
    ax.set_title(f"{mech.upper()} — missing rows highlighted")
    ax.set_xlabel("Row index (sorted by petal length)")
    ax.set_ylabel("Petal length")
    ax.legend(fontsize=8)

plt.suptitle("Iris petal length: which rows each MNAR mechanism removes (30%)", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "11_mnar_mechanisms.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/11_mnar_mechanisms.png")

## 3. Load Results

Run each dataset in a **separate terminal** (in parallel):
```
.venv/bin/python scripts/run_mnar_dataset.py Iris
.venv/bin/python scripts/run_mnar_dataset.py Glass
.venv/bin/python scripts/run_mnar_dataset.py Haberman
```
Each script saves `results/11_mnar_<dataset>.json`.
Re-run this cell after all three finish to load results.

In [ ]:
DATASET_NAMES = ["Iris", "Glass", "Haberman"]
results = {}

for ds_name in DATASET_NAMES:
    path = os.path.join(RESULTS_DIR, f"11_mnar_{ds_name.lower()}.json")
    if not os.path.exists(path):
        print(f"  [MISSING] {path} — run the script for {ds_name} first")
        continue
    with open(path) as f:
        data = json.load(f)
    results[ds_name] = data
    print(f"\n{ds_name}:")
    for mech in ["mar", "top", "bottom", "tails"]:
        m = data.get(mech, {})
        print(f"  {mech:8s}  RMSE={m['rmse']:.4f}  MAD={m['mad']:.4f}  CoD={m['cod']:.4f}")

print(f"\nLoaded {len(results)}/{len(DATASET_NAMES)} datasets.")

## 4. RMSE Comparison Table

In [ ]:
if results:
    print("=" * 70)
    print("RMSE — MIGA under MAR vs MNAR mechanisms (30% missing, l=200, G=500)")
    print("=" * 70)
    print(f"{'Dataset':<12}  {'MAR':>9}  {'top':>9}  {'bottom':>9}  {'tails':>9}")
    print("-" * 55)
    for ds_name in DATASET_NAMES:
        if ds_name not in results:
            print(f"  {ds_name:<12}  (not yet run)")
            continue
        data = results[ds_name]
        vals = [f"{data[m]['rmse']:.4f}" for m in ["mar", "top", "bottom", "tails"]]
        best_idx = np.argmin([data[m]["rmse"] for m in ["mar", "top", "bottom", "tails"]])
        vals[best_idx] = vals[best_idx] + " ★"
        print(f"  {ds_name:<12}  {'  '.join(v:>9 for v in vals)}")
    print("\n  ★ = lowest RMSE (best imputation) for this dataset")

## 5. Visualisation — RMSE by Mechanism and Dataset

In [ ]:
if results:
    mechs  = ["mar", "top", "bottom", "tails"]
    ds_list = [d for d in DATASET_NAMES if d in results]
    x = np.arange(len(ds_list))
    width = 0.2
    colours_m = ["tab:gray", "tab:red", "tab:blue", "tab:purple"]

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, (mech, col) in enumerate(zip(mechs, colours_m)):
        rmse_vals = [results[ds][mech]["rmse"] for ds in ds_list]
        ax.bar(x + (i - 1.5) * width, rmse_vals, width, label=mech, color=col, alpha=0.8)

    ax.set_xticks(x)
    ax.set_xticklabels(ds_list)
    ax.set_ylabel("NRMSE")
    ax.set_title("MIGA RMSE under MAR vs MNAR Mechanisms (30% missing)")
    ax.legend(title="Mechanism")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "11_rmse_by_mechanism.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: results/11_rmse_by_mechanism.png")

## 6. Convergence Plots (Fr vs Generation)

In [ ]:
if results:
    for ds_name, data in results.items():
        fig, axes = plt.subplots(1, 4, figsize=(18, 3.5), sharey=False)
        cols_m = {"mar": "tab:gray", "top": "tab:red", "bottom": "tab:blue", "tails": "tab:purple"}

        for ax, mech in zip(axes, ["mar", "top", "bottom", "tails"]):
            gen_hists = data[mech].get("gen_history", [])
            if not gen_hists:
                ax.text(0.5, 0.5, "no gen_history", transform=ax.transAxes, ha="center")
                continue
            arr = np.array(gen_hists)
            mean_curve = arr.mean(axis=0)
            std_curve  = arr.std(axis=0)
            gens = np.arange(arr.shape[1])
            ax.plot(gens, mean_curve, color=cols_m[mech], lw=2)
            ax.fill_between(gens, mean_curve - std_curve, mean_curve + std_curve,
                            alpha=0.15, color=cols_m[mech])
            ax.set_title(mech.upper())
            ax.set_xlabel("Generation")
            ax.set_ylabel("Best F_r")
            ax.grid(alpha=0.3)

        plt.suptitle(f"{ds_name} — Convergence under MAR vs MNAR", fontsize=13, y=1.02)
        plt.tight_layout()
        plt.savefig(os.path.join(RESULTS_DIR, f"11_{ds_name.lower()}_convergence.png"),
                    dpi=150, bbox_inches="tight")
        plt.show()

## 7. Discussion

### Key questions

1. **Does MIGA degrade under MNAR?**
   Compare RMSE(MAR) vs RMSE(top/bottom/tails) — if degradation is small,
   MIGA is robust to the missing mechanism.

2. **Which MNAR mechanism is hardest?**
   `top`/`bottom` create *systematic* biases in X_A vs X_C (available rows
   have lower/higher means than completed rows). `tails` removes extreme
   values, reducing X_A variance.

3. **Fr vs RMSE under MNAR**
   MIGA minimises distributional distance Fr.  Under MNAR, X_A and X_C
   may have structurally different distributions (e.g. X_C has only
   high-income individuals when mechanism=top). Fr cannot detect this
   structural asymmetry — it only measures how similar the *imputed*
   values' distribution is to X_A's distribution.

### Expected findings

- MIGA's Fr objective is distribution-agnostic: it asks "does X_C look
  like X_A?" Under MNAR, X_C genuinely differs from X_A in distribution,
  so a *low* Fr may still yield high RMSE.
- Distributional methods (MIGA) are expected to degrade less than
  regression-based methods (KNN, MICE) under MNAR — but this would require
  a baseline comparison notebook (future work).
- The `tails` mechanism may be harder than `top`/`bottom` because it
  removes the most informative boundary observations.